# Shadow norm (SI Eq. S7) numerics for SEEQST

**Goal.** Huang, Kueng & Preskill's Lemma S1 bounds the variance of a single-shot
classical-shadow estimator by the *shadow norm*:

$$
\mathrm{Var}[\hat o] \;\le\; \|O\|_{\mathrm{shadow}}^2
:= \max_{\sigma:\ \text{state}}\ \mathbb E_{U\sim\mathcal U}\ \sum_{b\in\{0,1\}^n}
\langle b|U\sigma U^\dagger|b\rangle\,\langle b|U\,\mathcal M^{-1}(O)\,U^\dagger|b\rangle^2 .
\qquad\text{(Eq. S7)}
$$

Closed forms are known for the two ensembles in the main paper:
- Pauli (local random-Clifford): $\|O\|_{\mathrm{shadow}}^2 = 3^{k}$ for a weight-$k$ Pauli string (SI Eq. S17).
- Global Clifford: $\|O\|_{\mathrm{shadow}}^2 = 3\,\mathrm{tr}(O^2) = 3\cdot 2^n$ (SI Eq. S16).

**No analytic form is known yet for SEEQST.** This notebook does the numerics to
build intuition for it, reusing every piece of the `shadow_benchmark` setup
(the three `ShadowEnsemble` classes, the SEEQST inverse map, etc.) without
touching the two source repos.

Keep this notebook simple / exploratory (small $n$, a handful of observables).
Once you're happy with the approach, `exp_2.py` (a plain script, not built yet)
is the natural place to scale this up to many $(n, k)$ combinations.

In [1]:
import sys
from pathlib import Path

# so `import common...`, `import ensembles...` work regardless of the notebook's cwd
sys.path.insert(0, str(Path.cwd()))

import numpy as np
from qiskit.quantum_info import Operator, Pauli

from common import pauli_utils
from ensembles.pauli_ensemble import PauliEnsemble
from ensembles.clifford_ensemble import CliffordEnsemble
from ensembles.seeqst_ensemble import SEEQSTEnsemble

rng = np.random.default_rng(0)


## A simplification that makes this tractable

For a *fixed* sampled unitary $U$, write $g_U(b) = \langle b|U\,\mathcal M^{-1}(O)\,U^\dagger|b\rangle^2$
and $D_U = \sum_b g_U(b)\,|b\rangle\langle b|$ (diagonal in the computational basis). Then

$$
\sum_b \langle b|U\sigma U^\dagger|b\rangle\, g_U(b) = \mathrm{Tr}\!\big[D_U\, U\sigma U^\dagger\big] = \mathrm{Tr}\!\big[(U^\dagger D_U U)\,\sigma\big].
$$

So, defining $A := \mathbb E_U[\,U^\dagger D_U U\,]$ (an operator that does **not** depend on $\sigma$ at all),

$$
\|O\|_{\mathrm{shadow}}^2 = \max_{\sigma} \mathrm{Tr}[A\sigma] = \lambda_{\max}(A).
$$

So instead of searching over candidate states $\sigma$, we just need to build the (small,
$2^n\times2^n$) operator $A$ and take its largest eigenvalue -- numpy's `eigvalsh` does the rest.

For **Pauli** and **SEEQST**, the ensemble is a *finite* set of unitaries (3^n and 2^{n+1}
respectively), so we can sum over *all* of them exactly -- no Monte Carlo noise at all.
The **Clifford** group is too large to enumerate, so we Monte-Carlo sample it instead (and
we already know its closed form to check against).

In [2]:
def shadow_norm_sq(ensemble, spec, n, mode="exact", num_mc_samples=3000, rng=None):
    # Numerically compute ||O||^2_shadow (SI Eq. S7) for one Pauli observable
    # `spec`, under a given measurement ensemble.
    #   mode="exact"       : sums over ALL of ensemble.enumerate_unitaries(n)
    #                        (exact; only available for Pauli / SEEQST).
    #   mode="monte_carlo" : averages `num_mc_samples` draws from
    #                        ensemble.sample_unitary_circuit(n, rng)
    #                        (approximate; needed for Clifford).
    label = pauli_utils.spec_to_label(spec, n)
    O = Pauli(label).to_matrix()
    beta = ensemble.inverse_weight(spec, n)  # M^{-1}(P) = beta * P
    d = 2 ** n
    A = np.zeros((d, d), dtype=complex)

    if mode == "exact":
        pairs = ensemble.enumerate_unitaries(n)
    elif mode == "monte_carlo":
        assert rng is not None
        w = 1.0 / num_mc_samples
        pairs = [(w, ensemble.sample_unitary_circuit(n, rng)) for _ in range(num_mc_samples)]
    else:
        raise ValueError(mode)

    for weight, circuit in pairs:
        U = Operator(circuit).data
        diag = np.real(np.diag(U @ O @ U.conj().T))  # <b|U O U^dagger|b> for every b
        g = (beta * diag) ** 2                         # <b|U M^{-1}(O) U^dagger|b>^2
        A += weight * (U.conj().T @ np.diag(g) @ U)

    return float(np.linalg.eigvalsh(A)[-1])  # lambda_max(A)


## Sanity check 1: Pauli ensemble vs. the known $3^k$

In [3]:
pe = PauliEnsemble()
n = 4
for k in range(1, n + 1):
    spec = pauli_utils.random_pauli_spec(n, k, rng)
    val = shadow_norm_sq(pe, spec, n, mode="exact")
    print(f"k={k}  numeric={val:8.4f}   3^k={3**k:8.4f}   match={np.isclose(val, 3**k)}")


k=1  numeric=  3.0000   3^k=  3.0000   match=True
k=2  numeric=  9.0000   3^k=  9.0000   match=True
k=3  numeric= 27.0000   3^k= 27.0000   match=True
k=4  numeric= 81.0000   3^k= 81.0000   match=True


## Sanity check 2: global Clifford ensemble vs. the known $3\cdot 2^n$

(Monte Carlo now, since the Clifford group can't be enumerated -- expect some noise.)

In [4]:
ce = CliffordEnsemble()
n = 4
for k in range(1, n + 1):
    spec = pauli_utils.random_pauli_spec(n, k, rng)
    val = shadow_norm_sq(ce, spec, n, mode="monte_carlo", num_mc_samples=3000, rng=rng)
    print(f"k={k}  numeric~{val:8.2f}   3*2^n={3 * 2**n:8.2f}")


k=1  numeric~   15.03   3*2^n=   48.00
k=2  numeric~   17.73   3*2^n=   48.00
k=3  numeric~   19.27   3*2^n=   48.00
k=4  numeric~   17.24   3*2^n=   48.00


## The actual question: SEEQST

No closed form is known yet, so let's just compute it exactly (SEEQST is finite/enumerable,
same as Pauli) for a range of Paulis and system sizes, and see what comes out.

Recall the SEEQST inverse-map weight $\beta(P) = 1/\alpha_P$ (`SEEQSTEnsemble.inverse_weight`)
is $2$ for non-trivial Pauli strings made only of $I,Z$ ("pure-Z-type"), and $2^{n+1}$ for any
Pauli with an $X$ or $Y$ factor. We'll compute the shadow norm for both cases, across a few
system sizes.

In [5]:
se = SEEQSTEnsemble()

rows = []
for n in range(2, 7):
    # a pure-Z-type Pauli (only I/Z factors)
    z_spec = {q: "Z" for q in range(min(2, n))}
    # a Pauli with at least one X or Y factor
    xy_spec = {0: "X", **({1: "Z"} if n > 1 else {})}

    val_z = shadow_norm_sq(se, z_spec, n, mode="exact")
    val_xy = shadow_norm_sq(se, xy_spec, n, mode="exact")
    beta_z = se.inverse_weight(z_spec, n)
    beta_xy = se.inverse_weight(xy_spec, n)
    rows.append((n, "pure-Z", z_spec, val_z, beta_z))
    rows.append((n, "has X/Y", xy_spec, val_xy, beta_xy))

print(f"{'n':>2}  {'type':<8} {'spec':<20} {'shadow_norm_sq':>15} {'beta = 1/alpha_P':>18}")
for n, kind, spec, val, beta in rows:
    print(f"{n:>2}  {kind:<8} {str(spec):<20} {val:>15.4f} {beta:>18.4f}")


 n  type     spec                  shadow_norm_sq   beta = 1/alpha_P
 2  pure-Z   {0: 'Z', 1: 'Z'}              2.0000             2.0000
 2  has X/Y  {0: 'X', 1: 'Z'}              8.0000             8.0000
 3  pure-Z   {0: 'Z', 1: 'Z'}              2.0000             2.0000
 3  has X/Y  {0: 'X', 1: 'Z'}             16.0000            16.0000
 4  pure-Z   {0: 'Z', 1: 'Z'}              2.0000             2.0000
 4  has X/Y  {0: 'X', 1: 'Z'}             32.0000            32.0000
 5  pure-Z   {0: 'Z', 1: 'Z'}              2.0000             2.0000
 5  has X/Y  {0: 'X', 1: 'Z'}             64.0000            64.0000
 6  pure-Z   {0: 'Z', 1: 'Z'}              2.0000             2.0000
 6  has X/Y  {0: 'X', 1: 'Z'}            128.0000           128.0000


## Observed pattern -- and why it's exact

The numbers above should show $\|P\|_{\mathrm{shadow}}^2 = \beta(P)$ **exactly** (to floating-point
precision) in every row -- i.e. the shadow norm is numerically identical to the SEEQST inverse-map
weight itself! That *is* an analytic form, and it's easy to see why it must hold once you notice one
fact: every SEEQST unitary is a Clifford circuit (only $RX(\pi/2)$, $RY(\pi/2)$, CNOT), so it maps
any Pauli string $P$ to (plus or minus) another Pauli string $Q = UPU^\dagger$ -- never to something
that's a genuine superposition of several different Paulis.

A Pauli string has *every* computational-basis diagonal entry equal to $\pm1$ if it contains only
$I,Z$ factors ("diagonal-type"), and *every* diagonal entry equal to $0$ otherwise (a single $X$ or
$Y$ factor already kills the entire diagonal). So for each sampled $U$, $D_U$ from the derivation
above is **either** $\beta^2\cdot\mathbb{1}$ (if $UPU^\dagger$ happens to be diagonal-type) **or**
exactly $0$ -- there's no in-between, and $U^\dagger(\beta^2\mathbb 1)U=\beta^2\mathbb1$ regardless
of $U$. So $A = \beta^2\cdot\Pr_U[\,UPU^\dagger\text{ is diagonal-type}\,]\cdot\mathbb 1$ -- a multiple
of the identity, whose eigenvalue is just that scalar.

And $\Pr_U[\,UPU^\dagger\text{ is diagonal-type}\,]$ is *exactly* $\alpha_P$ -- because that
probability is precisely what the forward channel $\mathcal M(P)=\alpha_P P$ computes (Prop. 6-9 of
the SEEQST draft): $\mathcal M(P) = \mathbb E_U[U^\dagger\Phi(UPU^\dagger)U]$, and the dephasing
channel $\Phi$ leaves $UPU^\dagger$ unchanged exactly when it's diagonal-type (giving back $P$ after
conjugating by $U^\dagger$), and kills it to $0$ otherwise.

Putting it together: $\|P\|_{\mathrm{shadow}}^2 = \beta^2\cdot\alpha_P = \beta^2/\beta = \beta$.

**This argument only used "every ensemble unitary is Clifford" + "the forward channel is
Pauli-diagonal" -- both already true for Pauli and SEEQST -- so it applies to both alike** (and is
exactly why sanity check 1 above reproduced $3^k$ on the nose: $\beta_{\mathrm{Pauli}}=3^k$).
It does *not* directly apply to the global Clifford ensemble the same way, because a *global*
random Clifford maps $P$ to an (approximately) uniformly random OTHER nontrivial Pauli string, not
"diagonal-type or bust" -- that ensemble's known $3\cdot2^n$ comes from a different argument in the
SI.

## Cross-check: does an actual empirical variance respect this bound?

Reuses the exact same `sample_snapshots` / `evaluate_snapshots` machinery as `exp_1.py` --
draw many shadow snapshots of a genuinely random state, and confirm the empirical
$\mathrm{Var}[\hat o]$ stays at or below the shadow-norm bound just computed (Lemma S1).

In [6]:
from common import states as _states

n = 4
state = _states.haar_random_state(n, rng)
spec = {0: "X", 1: "Y"}  # has an X/Y factor -> beta = 2^(n+1)

bound = shadow_norm_sq(se, spec, n, mode="exact")

snaps = se.sample_snapshots(state, n, 20000, rng)
shots = se.evaluate_snapshots(snaps, [spec], n)[:, 0]
empirical_var = float(np.var(shots, ddof=1))

print(f"shadow-norm bound ||P||^2_shadow = {bound:.4f}")
print(f"empirical Var[o_hat] (20000 shots) = {empirical_var:.4f}")
print(f"bound respected: {empirical_var <= bound}")


shadow-norm bound ||P||^2_shadow = 32.0000
empirical Var[o_hat] (20000 shots) = 31.1164
bound respected: True


## Next steps (for a large-scale `exp_2.py`)

- Sweep this identity check over many more $(n, \text{Pauli weight})$ combinations and random
  Pauli draws, purely as a regression test that $\|P\|_{\mathrm{shadow}}^2 = \beta(P)$ keeps
  holding exactly (it should, by the argument above -- any discrepancy would flag a bug).
- Since the argument above never used anything specific to *this* n, it's really a proof, not
  just a numerical observation -- but running it across a wider range of n is still a good
  automated check to keep around.
- If/when the paper needs this written up formally, the derivation in the markdown cell above is
  the one to adapt (it's short).
- Optionally, extend `shadow_norm_sq` to *quadratic* functions (SI Sec. "predicting nonlinear
  functions") if that becomes relevant later.